In [0]:
# =====================================================================================
# Notebook: 99_validation/03_silver_checker_v2.py
# Purpose : Validate Silver tables (freshness + contract + sanity)
# =====================================================================================

from pyspark.sql import functions as F

SILVER_TABLES = {
    "dim_team": "workspace.nba_silver.dim_team",
    "fact_game": "workspace.nba_silver.fact_game",
    "dim_date": "workspace.nba_silver.dim_date",
    "fact_game_enriched": "workspace.nba_silver.fact_game_enriched",
}

print("=== Row counts ===")
for name, t in SILVER_TABLES.items():
    df = spark.table(t)
    print(name, "->", t, "rows=", df.count(), "cols=", len(df.columns))

print("\n=== Freshness (fact_game) ===")
fg = spark.table(SILVER_TABLES["fact_game"])
fg.select(
    F.max("source_dt").alias("max_source_dt"),
    F.max("ingested_at").alias("max_ingested_at"),
    F.max("start_time_utc").alias("max_start_time_utc"),
    F.max("game_date_utc").alias("max_game_date_utc"),
).show(truncate=False)

print("\n=== Recent games (fact_game_enriched) ===")
fge = spark.table(SILVER_TABLES["fact_game_enriched"])
fge.select(
    "game_date_utc","start_time_utc","home_team_name","visitor_team_name",
    "home_score","visitor_score","status","game_state","season_start_year","source_dt"
).orderBy(F.col("start_time_utc").desc()).show(25, truncate=False)

print("\n=== game_state distribution ===")
fg.groupBy("game_state").count().orderBy(F.col("count").desc()).show(20, truncate=False)

print("\n=== Team name coverage (fact_game_enriched) ===")
fge.select(
    F.sum(F.when(F.col("home_team_name").isNull(), 1).otherwise(0)).alias("missing_home_team_name"),
    F.sum(F.when(F.col("visitor_team_name").isNull(), 1).otherwise(0)).alias("missing_visitor_team_name"),
    F.count("*").alias("total_games"),
).show(truncate=False)

print("\n=== dim_date coverage vs fact_game dates ===")
dd = spark.table(SILVER_TABLES["dim_date"])
dd_stats = dd.select(F.min("date_key").alias("min_date"), F.max("date_key").alias("max_date")).collect()[0]
fg_stats = fg.select(F.min("game_date_utc").alias("min_game_date"), F.max("game_date_utc").alias("max_game_date")).collect()[0]

print("dim_date range:", dd_stats["min_date"], "->", dd_stats["max_date"])
print("fact_game range:", fg_stats["min_game_date"], "->", fg_stats["max_game_date"])
